# Training a Transofmer based model with pl lighting

In [31]:
# Add import 
import sys
import torch 
from torch import nn
from torch import optim
from prodigyopt import Prodigy # proddigy optimizer from https://github.com/konstmish/prodigy?tab=readme-ov-file
import pytorch_lightning as pl
from torch.utils.data import DataLoader
import tqdm
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
# allow reload of python modules
%load_ext autoreload
%reload_ext autoreload
%autoreload 2
from dataset.RobotPathDataset.normalizer import MinMaxFeatureNormalizer

from model.models import EMA
import copy
import time

# remove all warnings
import warnings
warnings.filterwarnings("ignore")
from dataset import RobotPathDataset

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [32]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(device)
device

cuda:0


device(type='cuda', index=0)

In [33]:
normalizer = MinMaxFeatureNormalizer()
encoder_normalizer = MinMaxFeatureNormalizer()

In [34]:
epochs = 1000
representiation_dim = 64
EXPERIMENT_NAME='100 world 1000 path overfit offset-encoding end2end encoder trianing'

In [35]:

CONFIG = {
    # Model configuration
    'representation_dim': representiation_dim, # Dimension of the representation

    # specific model configuration
    'transfomer':{
        'num_heads':4,
        'use_sin_activation':True, 
        'num_layers': 4,
    },
    # U-Net Architecture
    'u_net' :{
        'dim_mults': (1, 2, 4, 8),  # Dimension multipliers for hidden layers of the model
        'attention': False,
    },

    # Embedder configuration
    'embeder_num_of_hidden_layers' : 1,
    'encoder_conditional_dim': representiation_dim//2,
    
    # Encoders
    # world encoder
    'world_encoder':{
        'end2end_train':False,
        'ckpt_path': r'/home/karim.samir.lotfy/adlr/tum-adlr-ss24-09/lightning_logs/curr_best/checkpoints/epoch=406-step=457875.ckpt',
    },
    # Hyperparameters for training
    'batch_size': 8,
    'num_epochs': epochs,
    'ema': EMA(beta=0.99), # Exponential moving average for the model weights
        ## Optimizer configuration
        'optimizer': Prodigy,
        'optimizer_kwargs': {
            'lr': 1., # ! ONLY FOR PRODIGY OPTIMIZER
            'weight_decay': 0.01, 
            'safeguard_warmup':True,
            'use_bias_correction':True,
            'betas': (0.9, 0.99),
            },
        # Scheduler configuration
        'scheduler': torch.optim.lr_scheduler.CosineAnnealingLR,
        'scheduler_kwargs': {
            # 'gamma': 0.999,
            'T_max': 10, # Total number of iterations
        },
    
    'loss_weights':{
        'MSE': 0.3,
        'ObstacleFreePathLoss': 0.5,
        'GraphBasedConsistencyLoss':0.2,
    },
    # Hyperparameters for diffusion process
    'noise_steps': 256,
    'normalize': True,
    'normalizer':normalizer,
    'encoder_normalizer':encoder_normalizer,
    'cfg_scale': 3,


    # Dataset specific configuration
    'n_paths_per_world': 10,
    'n_worlds': 10,
    'n_waypoints': 64, # due to the archtechture has to be a number that is a power of 2 
    'single_world_dataset':False,
}

In [36]:
u_net = {
    'dim_mults': (1, 2, 4, 8),  # Dimension multipliers for hidden layers of the model
    'attention': False,
}

In [37]:
file = '/home/karim.samir.lotfy/adlr/tum-adlr-ss24-09/data/SingleSphere02_all.db'
# file = '/home/karim.samir.lotfy/tum-adlr-ss24-09/data/SingleSphere02_one-world.db'
dataset = RobotPathDataset(file, n_paths_per_world=CONFIG['n_paths_per_world'], n_worlds=CONFIG['n_worlds'],  n_waypoints=CONFIG['n_waypoints'],   normalizer=CONFIG['normalizer'], single_world_dataset=CONFIG['single_world_dataset'])
print(f' sample shape {dataset[0]["path"].shape} with {len(dataset)} samples consisting of n worlds {len(np.unique(dataset.worlds_indx))}')
indx_sample = 5
sample = dataset[indx_sample];

data shape: torch.Size([100, 64, 2]), min_values: tensor([0, 0], device='cuda:0'), max_values: tensor([10, 10], device='cuda:0'), n_worlds:9, samples: 100
 sample shape torch.Size([64, 2]) with 100 samples consisting of n worlds 9


In [38]:
import torch.utils
import torch.utils.data


class RobotPathDataModule(pl.LightningDataModule):
    def __init__(self, dataset, batch_size: int = 64, val_dataset_ratio=0.01):
        super().__init__()
        self.dataset = dataset
        self.batch_size = batch_size
        self.val_dataset_ratio = val_dataset_ratio

    def setup(self, stage=None):
        # Assign train/val datasets for use in dataloaders
        if stage == 'fit' or stage is None:
            self.train_dataset, self.val_dataset = self._split_train_val(self.dataset)

        # Assign test dataset for use in dataloader(s)
        if stage == 'test' or stage is None:
            assert NotImplementedError('Test dataset is not implemented yet') 

    def train_dataloader(self):
        return DataLoader(self.train_dataset, batch_size=self.batch_size)

    def val_dataloader(self):
        return DataLoader(self.val_dataset, batch_size=self.batch_size)


    def _split_train_val(self, dataset):
        return torch.utils.data.random_split(dataset, [1-self.val_dataset_ratio, self.val_dataset_ratio])
    
    def single_sample(self, indx=None):
        if indx is None:
            indx = np.random.randint(0, len(self.dataset))
        return self.dataset[indx]
    
    def single_batch(self):
        return next(iter(self.train_dataloader()))
    
data = RobotPathDataModule(dataset, batch_size=CONFIG['batch_size'], val_dataset_ratio=0.01)
data.setup()

In [39]:
sample = data.single_sample()
sample.keys()

dict_keys(['path', 'og_path', 'world_indx', 'world_img', 'world_distance_field_img', 'offset_path', 'straight_line_path'])

In [40]:
# sin activation

class Sine(nn.Module): # Sine activation
    def __init__(self, w0 = 1.):
        super().__init__()
        self.w0 = w0
    def forward(self, x):
        return torch.sin(self.w0 * x)

class TransformerEncoder(nn.Module):
    def __init__(self, input_dim, output_dim, cond_dim ,d_model, nhead, num_layers, use_sin_activation=False):
        super(TransformerEncoder, self).__init__()
        self.d_model = d_model
        self.nhead = nhead
        self.num_layers = num_layers
         
        self.process_input = nn.Sequential(
            nn.Linear(input_dim, d_model),
            Sine() if use_sin_activation else nn.ReLU(),
        )
        
        self.timestep_encoder = nn.Sequential(
            nn.Linear(1, d_model),
            Sine() if use_sin_activation else nn.ReLU(),
        )
        
        self.cond_encoder = nn.Sequential(
            nn.Linear(cond_dim, d_model),
            Sine() if use_sin_activation else nn.ReLU(),
        )
        
        self.cond_norm = nn.LayerNorm(d_model) # replace with ada norm from https://github.com/facebookresearch/DiT/tree/main
        

        seqTransEncoderLayer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead)
        self.seqTransformer1 = nn.TransformerEncoder(seqTransEncoderLayer,num_layers=num_layers)
        self.seqTransformer2 = nn.TransformerEncoder(seqTransEncoderLayer,num_layers=num_layers)

        self.proces_output = nn.Sequential(
            nn.Linear(d_model, output_dim),
            # No activation layer as output is gusassian noise anyway
        )
    def forward(self, x, t,cond=None):
        # x shape is (batch, n_waypoints, transtion_dim=2)
        # cond shape is (batch, cond_dim)
        x = self.process_input(x) # maps transtion_dim to d_model
        x = self.seqTransformer1(x) # 1st transformer layer output (batch, n_waypoints, d_model)
        
        
        t = t.unsqueeze(1) # (batch) to (batch, 1)
        emb = self.timestep_encoder(t.type(torch.float32)) # maps timestep to d_model (batch, d_model)
        if cond is not None:
            cond = self.cond_encoder(cond) # maps cond_dim to d_model (batch, d_model)
            emb = emb + cond # (batch, d_model)
        emb = self.cond_norm(emb) # (batch, d_model)
        emb = emb.unsqueeze(1)
        x = x + emb 
        x = self.seqTransformer2(x)
        x = self.proces_output(x)
        return x


In [41]:
from model.diffusion import Diffusion
import pytorch_lightning as L
from model.encoder import VAEEncoder
from model.vae import VAEXperiment
from utils.path import encoded_path_to_real_path
from model import ObstacleFreePathLoss, GraphBasedConsistencyLoss

class PathDiffusionModel(L.LightningModule):
    def __init__(self, config_dict, sample_input_batch, encoder=None,path_type='offset_path', cond_type='world_distance_field_img'):
        super().__init__()
        
        self.config = config_dict
        
        sample_input = sample_input_batch[path_type]
        self.transition_dim = sample_input.shape[-1]
        self.representation_dim = config_dict['representation_dim']
        self.path_type = path_type
        self.cond_type = cond_type
        
        self.world_encoder = encoder
        
        # Loss functions
        self.obstacle_avoidance_loss_fn = ObstacleFreePathLoss(normalizer=self.config['normalizer'])
        self.graph_based_consistency_loss_fn = GraphBasedConsistencyLoss()
        
        
        # Create model
        self.model = TransformerEncoder(input_dim=self.transition_dim, output_dim=self.transition_dim, cond_dim=representiation_dim, d_model=self.representation_dim, nhead=CONFIG['transfomer']['num_heads'], num_layers=CONFIG['transfomer']['num_layers'], use_sin_activation=CONFIG['transfomer']['use_sin_activation'])

        # diffusion process
        self.diffusion = Diffusion(input_shape=sample_input.shape[1:], noise_steps=config_dict['noise_steps'])

        # Encoders
        # world encoder
        # Import pretrained vae model
        full_vae = VAEXperiment.load_from_checkpoint(self.config['world_encoder']['ckpt_path'])
        if self.config['world_encoder']['end2end_train'] == False:
            full_vae.freeze()
        vae_world_encoder = full_vae.model
        self.world_encoder = VAEEncoder(vae_world_encoder)
        # start_end position encoder input: (batch, transition_dim, n_waypoints) output: (batch, representation_dim, n_waypoints)
        self.start_end_encoder =  nn.Sequential(
            nn.Linear(self.transition_dim, self.representation_dim),
            Sine() if self.config['transfomer']['use_sin_activation'] else nn.ReLU(),
        )
        self.save_hyperparameters()
    
    def training_step(self, batch, batch_idx):
        # extract data
        # x_path: The path data extracted from the batch, typically a sequence of coordinates or actions.
        # x_world_indx: An index or identifier for the specific world or environment from which the path data is extracted.
        # x_world_img: An image representation of the world or environment associated with the path data in binary.
        # x_world_img_distance_field: A signed distance field image derived from x_world_img, representing distances within the environment.
        # x_offset_path: offset of each waypoint from the straight line path from start to end
        # x_straighline_path: A simplified or idealized version of the path, typically represented as a straight line from start to end point.
        x_path, x_world_indx, x_world_img, x_world_img_distance_field, x_offset_path, x_straigh_path = batch['path'], batch['world_indx'], batch['world_img'], batch['world_distance_field_img'], batch['offset_path'], batch['straight_line_path']
        # x: The selected path data for processing, determined by the 'path_type' specified in the batch. This could be any of the above path representations depending on the context.        
        x = batch[self.path_type]
        cond = batch[self.cond_type]
        start_pos = x[:, 0, :]
        end_pos = x[:, -1, :] 
        start_and_end_pos = x[:, [0, -1], :]
        # sample timesteps
        timesteps_choosen = self.diffusion.sample_timesteps(x.shape[0])
        x_t, noise = self.diffusion.noise_input(x,timesteps_choosen) 
        # encode conditionings
        # encoder world 
        world_encoding = self.world_encoder(cond)
        # start position encoding
        start_end_encoding = self.start_end_encoder(start_and_end_pos)
        
        # fuse the conditionings world encoding + start encoding + end encoding
        cond = world_encoding + start_end_encoding[:, 0,:] + start_end_encoding[:, 1,:]
        # get model to predict the noise
        pred_noise = self.model(x_t, timesteps_choosen, cond)
        # get predicted path depending on the noise
        pred_path = encoded_path_to_real_path(encoded_path=x, straight_path=x_straigh_path, path_type=self.path_type, start=start_pos, end=end_pos)
        loss_noise_prediction =nn.SmoothL1Loss()(noise, pred_noise)
        loss_obstacle_avoidance =self.obstacle_avoidance_loss_fn(pred_path, x_world_img_distance_field)
        loss_graph_based_consistency =self.graph_based_consistency_loss_fn(pred_path)
        loss = self.config['loss_weights']['MSE']*loss_noise_prediction + self.config['loss_weights']['ObstacleFreePathLoss']*loss_obstacle_avoidance +  self.config['loss_weights']['GraphBasedConsistencyLoss'] * loss_graph_based_consistency

        # log 
        self.log('train_loss', loss)
        self.log('train_loss/noise_prediction', loss_noise_prediction)
        self.log('train_loss/obstacle_avoidance', loss_obstacle_avoidance)
        self.log('train_loss/graph_based_consistency', loss_graph_based_consistency)
        return loss

    def configure_optimizers(self):
        optimizer = self.config['optimizer'](self.parameters(), **self.config['optimizer_kwargs'])
        scheduler = self.config['scheduler'](optimizer, **self.config['scheduler_kwargs'])
        return [optimizer], [scheduler]

model = PathDiffusionModel(CONFIG, sample, encoder=None, path_type='offset_path', cond_type='world_distance_field_img')

In [42]:
trainer = pl.Trainer(max_epochs=epochs, fast_dev_run=True)
trainer.fit(model, data)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
Running in `fast_dev_run` mode: will run the requested loop using 1 batch(es). Logging and checkpointing is suppressed.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]



  | Name                            | Type                      | Params | Mode 
--------------------------------------------------------------------------------------
0 | obstacle_avoidance_loss_fn      | ObstacleFreePathLoss      | 0      | train
1 | graph_based_consistency_loss_fn | GraphBasedConsistencyLoss | 0      | train
2 | model                           | TransformerEncoder        | 2.3 M  | train
3 | world_encoder                   | VAEEncoder                | 99.2 K | train
4 | start_end_encoder               | Sequential                | 192    | train
--------------------------------------------------------------------------------------
2.3 M     Trainable params
99.2 K    Non-trainable params
2.4 M     Total params
9.414     Total estimated model params size (MB)


Using decoupled weight decay


Training: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_steps=1` reached.
